In [ ]:
poi_raw = session.table(f"{DB}.{RAW}.STREAM_T_PERSON_ORG_INVOLVEMENT").filter(
    col("METADATA$ACTION") == "INSERT"
).drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID").cache_result()
print("Stream read complete")

In [ ]:
ind_cols = ["SEARCHED_IND"]
code_cols = ["REASON_CLOSED_CODE"]
user_cols = ["ADD_USER", "MOD_USER"]
trim_cols = ["SOURCE_FILE_NAME"]

all_cols = [c.name for c in poi_raw.schema.fields]
select_exprs = []
for c in all_cols:
    if c == "LOAD_TIMESTAMP":
        select_exprs.append(col(c).alias("RAW_LOAD_TIMESTAMP"))
    elif c in ind_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("N")).otherwise(trim(col(c))), lit("N")).alias(c))
    elif c in code_cols:
        select_exprs.append(upper(trim(col(c))).alias(c))
    elif c in user_cols:
        select_exprs.append(coalesce(when(trim(col(c)) == lit(""), lit("SYSTEM")).otherwise(trim(col(c))), lit("SYSTEM")).alias(c))
    elif c in trim_cols:
        select_exprs.append(trim(col(c)).alias(c))
    else:
        select_exprs.append(col(c))

poi_clean = poi_raw.select(select_exprs)
print("Transformations applied")

In [ ]:
valid_poi = poi_clean.filter(col("POI_ID").is_not_null())
invalid_poi = poi_clean.filter(col("POI_ID").is_null()).with_column("ERROR_REASON", lit("POI_ID_NULL"))
print("Valid/invalid split defined")

In [ ]:
checksum_cols = [
    ("POI_ID", "number"), ("SEARCHED_IND", "string"), ("REASON_CLOSED_CODE", "string"),
    ("ADD_USER", "string"), ("MOD_USER", "string"),
    ("INV_INV_ID", "number"), ("ADR_ADR_CALLED_FROM_ID", "number"),
    ("ORGN_ORGN_ID", "number"), ("PERSON_PERSON_ID", "number"),
    ("CAS_CAS_ID", "number"), ("IC_IC_ID", "number"),
    ("COMM_COMM_ID", "number"), ("TICKLER_ID", "number"),
    ("START_DATE", "timestamp"), ("END_DATE", "timestamp")
]
checksum_exprs = []
for col_name, col_type in checksum_cols:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

poi_final = valid_poi.with_column(
    "CHECKSUM", sha2(concat_ws(lit("|"), *checksum_exprs), 256)
).with_column("STAGING_LOADED_TIMESTAMP", current_timestamp())

poi_final.write.save_as_table(f"{DB}.{STG}.{STG_PERSON_ORG_INVOLVEMENT}", mode="truncate")
print(f"POI loaded into {STG}.{STG_PERSON_ORG_INVOLVEMENT}")

In [ ]:
invalid_count = invalid_poi.count()
if invalid_count > 0:
    invalid_poi.select(
        lit("T_PERSON_ORG_INVOLVEMENT").alias("TABLE_NAME"), col("ERROR_REASON"),
        col("SOURCE_FILE_NAME"), col("RAW_LOAD_TIMESTAMP")
    ).write.save_as_table(f"{DB}.{STG}.{INVALID_RECORDS}", mode="append")
    print(f"Invalid records saved: {invalid_count}")
else:
    print("No invalid records")

In [ ]:
rows_processed = session.table(f"{DB}.{STG}.{STG_PERSON_ORG_INVOLVEMENT}").count()
status = 'SUCCESS' if invalid_count == 0 else 'PARTIAL_SUCCESS'

session.call(f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    str(uuid.uuid4()), 'STG_PERSON_ORG_INVOLVEMENT_LOAD', 'STAGING',
    datetime.now(), datetime.now(), rows_processed, invalid_count, status, None)
session.call(f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status, 'STG_PERSON_ORG_INVOLVEMENT_LOAD', 'STAGING',
    f'POI staging completed. Rows: {rows_processed}, Failed: {invalid_count}')
print(f"STG_POI complete | Valid: {rows_processed} | Invalid: {invalid_count}")